# VT-Guard: huấn luyện từ manifest đã duyệt trên Colab
Notebook chỉ điều phối module trong repository. Điền GitHub URL và full commit SHA. Manifest cùng raw được duyệt phải nằm trên Drive. Checkpoint và bundle lưu bền trên Drive; dữ liệu được stage tạm trên /content.

In [ ]:
import os, subprocess, sys
from pathlib import Path
REPO_URL = ''  # Điền GitHub URL sau khi repo được tạo
CODE_COMMIT = ''  # Full commit SHA, không dùng nhánh di động
if not REPO_URL or len(CODE_COMMIT) != 40:
    raise ValueError('Cần GitHub URL và full commit SHA trước khi dùng Colab')
repo = Path('/content/vt-guard')
subprocess.run(['git', 'clone', '--filter=blob:none', REPO_URL, str(repo)], check=True)
subprocess.run(['git', '-C', str(repo), 'checkout', CODE_COMMIT], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-e', str(repo)], check=True)
os.environ['VTGUARD_CODE_COMMIT'] = CODE_COMMIT
print('Python:', sys.version)


In [ ]:
# Kiểm tra runtime và smoke CPU tổng hợp trước dữ liệu thật.
import torch
version = tuple(int(part) for part in torch.__version__.split('+')[0].split('.')[:2])
if version < (2, 7) or version[0] != 2:
    raise RuntimeError(f'PyTorch {torch.__version__} chưa được kiểm tra')
print('Python:', sys.version, 'PyTorch:', torch.__version__, 'CUDA:', torch.cuda.is_available())
subprocess.run([sys.executable, str(repo / 'scripts/smoke_training.py')], check=True)


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
MANIFEST = Path('/content/drive/MyDrive/vtguard/reviewed_manifest.json')
OUTPUT_ROOT = Path('/content/drive/MyDrive/vtguard/experiments/v1')
STAGED = Path('/content/vtguard_staged')
if not MANIFEST.is_file():
    raise FileNotFoundError('Thiếu reviewed manifest và các tệp raw tương ứng trên Drive')
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
subprocess.run([sys.executable, str(repo / 'scripts/stage_reviewed.py'),
                '--manifest', str(MANIFEST), '--output-dir', str(STAGED)], check=True)
STAGED_MANIFEST = STAGED / 'reviewed_manifest.json'
subprocess.run([sys.executable, str(repo / 'scripts/inspect_reviewed.py'),
                '--manifest', str(STAGED_MANIFEST),
                '--output', str(OUTPUT_ROOT / 'readiness.json')], check=True)
def run(script, *args):
    subprocess.run([sys.executable, str(repo / 'scripts' / script), *map(str, args)], check=True)
run('train_baseline.py', '--manifest', STAGED_MANIFEST, '--output', OUTPUT_ROOT / 'logistic')
run('train_hybrid.py', '--manifest', STAGED_MANIFEST, '--output', OUTPUT_ROOT / 'hybrid_logistic')
for kind in ('gru', 'gru_attention'):
    checkpoint = OUTPUT_ROOT / 'checkpoints' / kind
    args = ['--manifest', STAGED_MANIFEST, '--output', OUTPUT_ROOT / kind,
            '--checkpoint-dir', checkpoint, '--model-kind', kind, '--device', 'auto']
    if (checkpoint / 'checkpoint.pt').is_file():
        args.append('--resume')
    run('train_sequence.py', *args)
run('fit_sequence_hybrid.py', '--manifest', STAGED_MANIFEST,
    '--sequence-bundle', OUTPUT_ROOT / 'gru_attention',
    '--output', OUTPUT_ROOT / 'hybrid_gru_attention')
run('compare_experiments.py', '--root', OUTPUT_ROOT,
    '--output', OUTPUT_ROOT / 'experiment_report.json')
print('Đã xuất bundle thử nghiệm vào', OUTPUT_ROOT)
